In [1]:
import re
import time
import abc
import warnings
import json
import os
import functools
import subprocess
from pathlib import Path

import torch
import torchgeo
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Optional, Tuple, cast, Union, Iterator
from rtree.index import Index, Property
import pandas as pd
import geopandas as gpd
import rasterio
import timm

import torch.nn.functional as F
import torchvision.transforms as Tr
from torch.optim import Adam
from transformers import AutoVideoProcessor, AutoModel
from kornia.augmentation import AugmentationSequential
from torch.utils.data import DataLoader, Sampler
from torchgeo.datasets import RasterDataset, stack_samples, BoundingBox, GeoDataset
from torchgeo.samplers import GridGeoSampler, RandomGeoSampler
from torchgeo.samplers.constants import Units
from torchgeo.samplers.utils import _to_tuple, get_random_bounding_box, tile_to_chips

from utils.train_utils import *
os.chdir("D:/githubs/vjepa2")
#os.chdir(os.path.join(os.getcwd(), ".."))
import src.datasets.utils.video.transforms as video_transforms
import src.datasets.utils.video.volume_transforms as volume_transforms
from src.models.attentive_pooler import AttentiveClassifier
from src.models.vision_transformer import vit_giant_xformers_rope


from src.Prithvi.Prithvi_run_inference import MaskedAutoencoderViT
from DOFA.models_dwv import vit_base_patch16

d:\tulet\envs\torch_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\tulet\envs\torch_gpu\Lib\site-packages\kornia\feature\lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
d:\tulet\envs\torch_gpu\Lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
IMAGENET_DEFAULT_MEAN = (0.485*255, 0.456*255, 0.406*255)
IMAGENET_DEFAULT_STD = (0.229*255, 0.224*255, 0.225*255)

In [ ]:
Nb_dates = 6
extensions = ('.tif', '.tiff', '.TIF', '.TIFF')
img_dir = "D:/Phenologie/DL_Phenophases/data/Paracou/img"
img_files = sorted([file for file in os.listdir(img_dir) if file.endswith(extensions) and "mask" not in file])
print(img_files)
dates_data_df = pd.DataFrame({'img':img_files})

start = int(Nb_dates/2)
end = len(dates_data_df) - int((Nb_dates-1)/2)

In [ ]:
"""
# HuggingFace model repo name
hf_model_name = (
    "facebook/vjepa2-vitg-fpc64-384"  # Replace with your favored model, e.g. facebook/vjepa2-vitg-fpc64-384
)
# Path to local PyTorch weights
pt_model_path = "YOUR_MODEL_PATH"

# Initialize the HuggingFace model, load pretrained weights
model_hf = AutoModel.from_pretrained(hf_model_name)
model_hf.cuda().eval()
"""

def load_pretrained_vjepa_pt_weights(model, pretrained_weights):
    # Load weights of the VJEPA2 encoder
    # The PyTorch state_dict is already preprocessed to have the right key names
    pretrained_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")["encoder"]
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    pretrained_dict = {k.replace("backbone.", ""): v for k, v in pretrained_dict.items()}
    msg = model.load_state_dict(pretrained_dict, strict=False)
    print("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))


def load_pretrained_vjepa_classifier_weights(model, pretrained_weights):
    # Load weights of the VJEPA2 classifier
    # The PyTorch state_dict is already preprocessed to have the right key names
    pretrained_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")["classifiers"][0]
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    msg = model.load_state_dict(pretrained_dict, strict=False)
    print("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))


def build_pt_video_transform(img_size):
    short_side_size = int(256.0 / 224 * img_size)
    # Eval transform has no random cropping nor flip
    eval_transform = video_transforms.Compose(
        [
            video_transforms.Resize(short_side_size, interpolation="bilinear"),
            video_transforms.CenterCrop(size=(img_size, img_size)),
            volume_transforms.ClipToTensor(),
            video_transforms.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD),
        ]
    )
    return eval_transform


def forward_vjepa(imgs, model):
    # Run a sample inference with VJEPA
    print(imgs.shape)
    with torch.inference_mode():
        # Read and pre-process the image
        # video = get_video()  # T x H x W x C
        # video = torch.from_numpy(video).permute(0, 3, 1, 2)  # T x C x H x W
        # x = transform(video).cuda().unsqueeze(0)
        # Extract the patch-wise features from the last layer
        #imgs = torch.einsum('btchw->bcthw', imgs)  # B x T x C x H x W -> B x C x T x H x W
        out_patch_features = model(imgs)
        #out_patch_features_hf = model_hf.get_vision_features(x_hf)

    return out_patch_features


def get_vjepa_classification_results(classifier, out_patch_features):
    PHENO_CLASSES = json.load(open("pheno_classes.json", "r"))

    with torch.inference_mode():
        out_classifier = classifier(out_patch_features)
    preds = out_classifier.argmax(1)
    for pred in preds.detach().cpu().numpy():
        print(PHENO_CLASSES[str(pred)])

    return

#### Prithvi multiple dates

In [21]:
def forward_vjepa(imgs, model):
    print(imgs.shape)
    with torch.inference_mode():
        imgs = torch.einsum('btchw->bcthw', imgs)  # B x T x C x H x W -> B x C x T x H x W
        out_patch_features = model(imgs)
        #out_patch_features_hf = model_hf.get_vision_features(x_hf)

    return out_patch_features

In [ ]:
targSize = 224
tile_size = 112
sep_dates = True

decoder_depth=8
decoder_embed_dim=512
decoder_num_heads=16
depth=12
embed_dim=768
in_chans=3  #len(band_indices) if band_indices is not None else 3
num_frames=Nb_dates
num_heads=12
patch_size=16
tubelet_size=2

encoder = MaskedAutoencoderViT(
            img_size=targSize,
            patch_size=patch_size,
            num_frames=num_frames,
            tubelet_size=tubelet_size,
            in_chans=in_chans,
            embed_dim=embed_dim,
            depth=depth,
            num_heads=num_heads,
            decoder_embed_dim=decoder_embed_dim,
            decoder_depth=decoder_depth,
            decoder_num_heads=decoder_num_heads,
            mlp_ratio=4.0,
            norm_layer=functools.partial(torch.nn.LayerNorm, eps=1e-6),
            norm_pix_loss=False,
        )
state_dict = torch.load("D:/githubs/Prithvi_100M/Prithvi_100M.pt")
del state_dict["pos_embed"]
del state_dict["decoder_pos_embed"]
del state_dict['patch_embed.proj.weight']
del state_dict['patch_embed.proj.bias']
del state_dict["decoder_pred.weight"]
del state_dict["decoder_pred.bias"]
encoder.load_state_dict(state_dict, strict = False)
encoder.cuda()

classifier = (
    AttentiveClassifier(embed_dim=encoder.decoder_pred.out_features, num_heads=16, depth=4, num_classes=4).cuda().eval()
)

In [ ]:
transforms=DictTransform(AugmentationSequential(T.Resize((targSize, targSize)), T.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD), data_keys=["image"]))

for k in range(start, start+1): #end
    filepaths = [os.path.join(img_dir, dates_data_df['img'][i]) for i in range(k-int(Nb_dates/2), k+int((Nb_dates+1)/2))]
    print("filepaths : ", filepaths)
    inf_dataset = MultiDateRGBDataset(filepaths, transforms=transforms, sep_dates=False)

sampler = RandomGeoSampler(inf_dataset, size=tile_size, length=10)
dataloader = DataLoader(inf_dataset, batch_size=1, sampler=sampler, collate_fn=stack_samples, shuffle=False)
for batch in dataloader:
    img = batch['image'].cuda()
    feat = forward_vjepa(img, encoder)       #out = B x num_patches x embed_dim = B x 14*14*nb_dates/tubelet_size x 1536 = B x 588 x 1536
    #print(feat[1].shape)
    get_vjepa_classification_results(classifier, feat[1])
    

#### Dinov2 Single date

In [13]:
def forward_vjepa(imgs, model):
    with torch.inference_mode():
        out_patch_features = model.forward_features(imgs)
        #out_patch_features_hf = model_hf.get_vision_features(x_hf)

    return out_patch_features

In [14]:
img_dir = "Z:/shared/PhenOBS_Africa/Mbalmayo/Mosa_rect_Oct2025"
targSize = 518
tile_size = 74

transforms = DictTransform(AugmentationSequential(T.Resize((targSize, targSize)), T.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD), data_keys=["image"]))

encoder = timm.create_model('vit_base_patch14_dinov2', pretrained=True)
encoder.cuda()

classifier = (
    AttentiveClassifier(embed_dim=encoder.num_features, num_heads=16, depth=4, num_classes=4).cuda().eval()
)
classifier.cuda()

AttentiveClassifier(
  (pooler): AttentivePooler(
    (cross_attention_block): CrossAttentionBlock(
      (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (xattn): CrossAttention(
        (q): Linear(in_features=768, out_features=768, bias=True)
        (kv): Linear(in_features=768, out_features=1536, bias=True)
      )
      (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (blocks): ModuleList(
      (0-2): 3 x Block(
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, b

In [ ]:
files_list = [file for file in os.listdir(img_dir) if file.endswith(extensions)]
print(files_list)
for f in files_list[:1]:
    print(f)
    inf_dataset = RobustGeoDataset(RasterDataset(paths=[os.path.join(img_dir, f)], transforms=transforms))
    sampler = RandomGeoSampler(inf_dataset, size=tile_size, length=10)
    dataloader = DataLoader(inf_dataset, batch_size=1, sampler=sampler, collate_fn=stack_samples, shuffle=False)
    for batch in dataloader:
        img = batch['image'].cuda()
        feat = forward_vjepa(img, encoder)       #out = B x num_patches x embed_dim = B x 14*14 x 768
        print(feat.shape)
        get_vjepa_classification_results(classifier, feat[:, 1:])

#### Embeddings pré-calculation

In [6]:
shp_path = "D:/Phenologie/DL_Phenophases/data/Mbalmayo/Mbalmayo_crowns_2024_04_05_32632.gpkg"
crowns_shp = gpd.read_file(shp_path)
for i in range(len(crowns_shp)):
    if crowns_shp['geometry'][i].geom_type == 'MultiPolygon':
        crowns_shp['geometry'][i] = crowns_shp['geometry'][i].geoms[0]

list_rois = [polygon_to_bbox(poly) for poly in crowns_shp['geometry']]

extensions = ('.tif', '.tiff', '.TIF', '.TIFF')
img_dir = "Z:/shared/PhenOBS_Africa/Mbalmayo/Mosa_rect_Oct2025"
Mbal24_dates = [f for f in os.listdir(img_dir) if f.endswith(extensions)][34:58]

##### DOFA

In [3]:
targSize = 224
tile_size = 112
wavelengths = [0.665, 0.56, 0.49]

transforms = DictTransform(AugmentationSequential(Tr.Resize((targSize, targSize)), Tr.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD), data_keys=["image"]))

encoder = vit_base_patch16(global_pool=False, img_size=224)
encoder.head = nn.Identity()
encoder.cuda()

OFAViT(
  (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
  (patch_embed): Dynamic_MLP_OFA(
    (weight_generator): TransformerWeightGenerator(
      (transformer_encoder): TransformerEncoder(
        (layers): ModuleList(
          (0): TransformerEncoderLayer(
            (self_attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
            )
            (linear1): Linear(in_features=128, out_features=2048, bias=True)
            (dropout): Dropout(p=False, inplace=False)
            (linear2): Linear(in_features=2048, out_features=128, bias=True)
            (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (dropout1): Dropout(p=False, inplace=False)
            (dropout2): Dropout(p=False, inplace=False)
          )
        )
      )
      (fc_weight): Linear(in_features=128, out_features=196

##### Dinov2

In [74]:
targSize = 518
tile_size = 74

transforms = DictTransform(AugmentationSequential(T.Resize((targSize, targSize)), T.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD), data_keys=["image"]))

encoder = timm.create_model('vit_base_patch14_dinov2', pretrained=True)
encoder.cuda()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=Fal

##### Processing

In [ ]:
out_dir = "Z:/users/HadrienTulet/DL_pheno/vjepa/dinov2_Mbal25_clstokens"

print(Mbal24_dates)
for file in Mbal24_dates:
    print(file)
    inf_dataset = RobustGeoDataset(RasterDataset(paths=[os.path.join(img_dir, file)], transforms=transforms))
    sampler = MultipleGridGeoSampler(inf_dataset, size=tile_size, stride=tile_size, roi_bounds=list_rois)
    dataloader = DataLoader(inf_dataset, batch_size=1, sampler=sampler, collate_fn=skip_none_collate, shuffle=False)
    date = file.split("_")[0]
    N = len(dataloader)
    blank = np.zeros((N, int((targSize/encoder.patch_embed.patch_size[0])**2), 768), dtype=np.float16)          #encoder.num_features
    print(blank.shape)
    for i, batch in enumerate(dataloader):
        if batch is None:
            continue
        else:
            print(f"{i/N:.2%}", end="\r")
            bboxes = batch['bbox']
            img = batch['image'].cuda()
            with torch.inference_mode():
                feat = encoder.forward_features(img, wave_list = wavelengths)                #out = B x num_patches x embed_dim = B x 14*14 x 768
                feat = feat.detach().cpu().numpy().astype(np.float16).squeeze(0)
                blank[i] = feat[1:]
    np.save(os.path.join(out_dir, f"{date}_features.npy"), blank)

['20240111_Mbalmayo_ORTHO_aligned_local.tif', '20240126_Mbalmayo_ORTHO_aligned_local.tif', '20240212_Mbalmayo_ORTHO_aligned_local.tif', '20240227_Mbalmayo_ORTHO_aligned_local.tif', '20240313_Mbalmayo_ORTHO_aligned_local.tif', '20240328_Mbalmayo_ORTHO_aligned_local.tif', '20240413_Mbalmayo_ORTHO_aligned_local.tif', '20240427_Mbalmayo_ORTHO_aligned_local.tif', '20240513_Mbalmayo_ORTHO_aligned_local.tif', '20240527_Mbalmayo_ORTHO_aligned_local.tif', '20240613_Mbalmayo_ORTHO_aligned_local.tif', '20240627_Mbalmayo_ORTHO_aligned_local.tif', '20240712_Mbalmayo_ORTHO_aligned_local.tif', '20240727_Mbalmayo_ORTHO_aligned_local.tif', '20240812_Mbalmayo_ORTHO_aligned_local.tif', '20240827_Mbalmayo_ORTHO_aligned_local.tif', '20240912_Mbalmayo_ORTHO_aligned_local.tif', '20240928_Mbalmayo_ORTHO_aligned_local.tif', '20241014_Mbalmayo_ORTHO_aligned_local.tif', '20241028_Mbalmayo_ORTHO_aligned_local.tif', '20241112_Mbalmayo_ORTHO_aligned_local.tif', '20241127_Mbalmayo_ORTHO_aligned_local.tif', '20241212

#### Reshape features spatially

In [ ]:
feat_dir = "Z:/users/HadrienTulet/DL_pheno/vjepa/dinov2_Mbal25_patchtokens"
output_dir = Path("Z:/users/HadrienTulet/DL_pheno/vjepa/dinov2_Mbal25_patchtokens_reshape")
k = 23

input_dir  = Path(feat_dir)
output_dir.mkdir(parents=True, exist_ok=True)

arrays = [np.load(os.path.join(input_dir, f), mmap_mode='r') for f in os.listdir(input_dir)]

N = arrays[0].shape[0]
assert all(a.shape[0] == N for a in arrays), "All dates must have the same N"
assert N % k == 0, f"N={N} is not evenly divisible by k={k}"

chunk_size = N // k
print(f"N={N}, k={k}, chunk_size={chunk_size}")

for i in range(2, k):
    start, end = i * chunk_size, (i + 1) * chunk_size

    chunk = np.stack([a[start:end] for a in arrays], axis=0)

    out_path = output_dir / f"chunk_{i}_features.npy"
    np.save(out_path, chunk)
    print(f"  Saved {out_path}  shape={chunk.shape}")

print("Done.")

#### Auto-supervised training

In [ ]:
FEAT_DIR   = "features/"       # directory containing .npy feature files
CKPT_PATH  = "predictor.pt"    # where to save checkpoints
T          = 4                  # number of input frames fed to the aggregator
BATCH_SIZE = 32                 # number of tiles per batch (must be << N ~1000)
N_EPOCHS   = 20
LR         = 1e-4
N_WORKERS  = 4                  # DataLoader worker processes
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_loop(classifier: nn.Module, train_files: list, val_files:list, feat_dir: str = FEAT_DIR):

    optimizer = torch.optim.AdamW(classifier.parameters(), lr=LR)
    loss_fn   = nn.SmoothL1Loss()
    classifier.to(DEVICE)

    for epoch in range(N_EPOCHS):

        classifier.train()
        epoch_loss  = 0.0
        n_steps     = 0

        for f in tqdm(train_files, desc=f"Epoch {epoch+1}/{N_EPOCHS}"):

            # -- load pre-extracted DOFA features ----------------------------
            # expected shape on disk: (N_patches, T_total, D) or similar;
            # after transpose: (T_total, N_patches, D)
            feat_data = np.load(os.path.join(feat_dir, f))
            feat_data = torch.tensor(
                np.transpose(feat_data, (1, 0, 2, 3)),   # → (T_total, N_patches, D)
                dtype=torch.float32,
            )

            T_total = feat_data.shape[0]

            # -- slide a window of size T+1 over the time axis ---------------
            for t in range(T_total - T):

                vec     = feat_data[t : t + T]      #(T, N_patches, D)
                tgt_vec = feat_data[t + T]          #(N_patches, D)

                # add batch dim: (1, T, N_patches, D) and (1, N_patches, D)
                vec     = vec.unsqueeze(0).to(DEVICE)
                tgt_vec = tgt_vec.unsqueeze(0).to(DEVICE)

                # stop-gradient on target: DOFA encoder must not drift
                with torch.no_grad():
                    target = tgt_vec.detach()

                # forward + loss
                optimizer.zero_grad()
                predicted = classifier(vec)                   # (1, N_patches, D)
                loss      = loss_fn(predicted, target)

                # backward + step
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()
                n_steps    += 1

        avg_loss = epoch_loss / max(n_steps, 1)

        classifier.eval()
        for f in tqdm(val_files, desc=f"Epoch {epoch+1}/{N_EPOCHS}"):

            # -- load pre-extracted DOFA features ----------------------------
            # expected shape on disk: (N_patches, T_total, D) or similar;
            # after transpose: (T_total, N_patches, D)
            feat_data = np.load(os.path.join(feat_dir, f))
            feat_data = torch.tensor(
                np.transpose(feat_data, (1, 0, 2, 3)),   # → (T_total, N_patches, D)
                dtype=torch.float32,
            )

            T_total = feat_data.shape[0]

            # -- slide a window of size T+1 over the time axis ---------------
            for t in range(T_total - T):

                vec     = feat_data[t : t + T]      # (T, N_patches, D)  — input
                tgt_vec = feat_data[t + T]          # (N_patches, D)     — target

                # add batch dim: (1, T, N_patches, D) and (1, N_patches, D)
                vec     = vec.unsqueeze(0).to(DEVICE)
                tgt_vec = tgt_vec.unsqueeze(0).to(DEVICE)

                # stop-gradient on target: DOFA encoder must not drift
                with torch.no_grad():
                    target = tgt_vec.detach()

                # forward + loss
                optimizer.zero_grad()
                predicted = classifier(vec)                   # (1, N_patches, D)
                loss      = loss_fn(predicted, target)


                epoch_loss += loss.item()
                n_steps    += 1

        avg_loss = epoch_loss / max(n_steps, 1)
        print(f"Epoch {epoch+1}/{N_EPOCHS} — avg loss: {avg_loss:.6f}")

        # -- checkpoint every epoch -----------------------------------------
        torch.save({
            "epoch":      epoch + 1,
            "model":      classifier.state_dict(),
            "optimizer":  optimizer.state_dict(),
            "avg_loss":   avg_loss,
        }, CKPT_PATH)

In [4]:
loss_fn = torch.nn.L1Loss()

In [5]:
class TemporalAggregator(nn.Module):
    def __init__(self, embed_dim, num_heads=8, num_layers=2):
        super().__init__()
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim))  # learned aggregation token

    def forward(self, x):  # x: (B, T, N, D)
        B, T, N, D = x.shape
        # Process each spatial token's temporal sequence independently
        x = x.permute(0, 2, 1, 3).reshape(B * N, T, D)  # (B*N, T, D)
        
        # Prepend learned query token
        q = self.query.expand(B * N, -1, -1)             # (B*N, 1, D)
        x = torch.cat([q, x], dim=1)                     # (B*N, T+1, D)
        
        x = self.transformer(x)                          # (B*N, T+1, D)
        x = x[:, 0]                                      # take query token → (B*N, D)
        
        return x.reshape(B, N, D)                        # (B, N, D)

In [6]:
"""
classifier = (
    AttentiveClassifier(embed_dim=768, num_heads=16, depth=4).cuda().eval()
)
classifier.linear = nn.Identity()
classifier = classifier.half()
classifier = classifier.cuda()
"""
classifier = TemporalAggregator(embed_dim = 768, num_heads=16, num_layers=4).cuda().eval()
classifier

TemporalAggregator(
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=768, bias=True)
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
)

In [ ]:
import abc
import os
from tqdm import tqdm
 
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import List, Optional, Tuple, cast, Union, Iterator
from rtree.index import Index, Property
import rasterio

from torch.utils.data import DataLoader, Sampler, Dataset
from torch.optim import Adam
from torchgeo.datasets import RasterDataset, stack_samples, BoundingBox, GeoDataset
from torchgeo.samplers.constants import Units
from torchgeo.samplers.utils import _to_tuple, get_random_bounding_box, tile_to_chips

os.chdir("D:/githubs/vjepa2")

class TileWindowDataset(Dataset):
    """
    Loads one .npy file of shape (T_total, N, N_patches, embed_dim) and
    exposes every (tile, time_window) pair as an individual sample.

    Each sample is:
        vec     — (T, N_patches, embed_dim)  input window
        target  — (N_patches, embed_dim)     frame immediately after the window
        tile_idx — int, for debugging / analysis
    """

    def __init__(self, filepath: str, T: int):
        # feat_data: (T_total, N, N_patches, embed_dim)
        self.data = torch.from_numpy(
            np.load(filepath),
            dtype=torch.float32,
        )
        print(f"{filepath} : {self.data.shape}")
        self.T       = T
        self.T_total = self.data.shape[0]
        self.N       = self.data.shape[1]

        # build flat index: list of (tile_idx, t) pairs
        self.indices = [
            (n, t)
            for n in range(self.N)
            for t in range(self.T_total - self.T)
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        n, t = self.indices[idx]
        vec    = self.data[t : t + self.T, n]       # (T, N_patches, embed_dim)
        target = self.data[t + self.T, n]           # (N_patches, embed_dim)
        return vec, target

def run_epoch(classifier, loader, optimizer, loss_fn, train: bool):

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    classifier.train() if train else classifier.eval()
    total_loss, n_steps = 0.0, 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for vec, target in tqdm(loader):
            # vec:    (B, T, N_patches, embed_dim)
            # target: (B, N_patches, embed_dim)
            vec    = vec.to(DEVICE)
            target = target.to(DEVICE).detach()     # stop-gradient on target

            predicted = classifier(vec)             # (B, N_patches, embed_dim)
            loss      = loss_fn(predicted, target)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            n_steps    += 1

    return total_loss / n_steps

def train_loop(classifier: nn.Module, feat_dir: str, out_path: str, T: int, loss_fn: nn.Module, batch_size: int, N_epochs: int, lr: float, n_workers: int = 1):

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    features_files = os.listdir(feat_dir)
    sep = int(np.ceil(3*len(features_files)/4))
    train_files = features_files[:sep]
    val_files = features_files[sep:]
    print("Train files : \n", train_files, "\nVal files : \n", val_files)
    
    optimizer = Adam(classifier.parameters(), lr=lr)
    classifier.to(DEVICE)

    for epoch in range(N_epochs):
        print(f"\nEpoch {epoch+1}/{N_epochs}")

        # Training
        train_loss = 0.0
        for f in tqdm(train_files, desc="train files"):
            dataset = TileWindowDataset(os.path.join(feat_dir, f), T=T)
            loader  = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=True,
                num_workers=n_workers,
                pin_memory=True,
            )
            train_loss += run_epoch(classifier, loader, optimizer, loss_fn, train=True)
        train_loss /= max(len(train_files), 1)

        # Val
        val_loss = 0.0
        for f in tqdm(val_files, desc="val files"):
            dataset = TileWindowDataset(os.path.join(feat_dir, f), T=T)
            loader  = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=False,
                num_workers=n_workers,
                pin_memory=True,
            )
            val_loss += run_epoch(classifier, loader, optimizer, loss_fn, train=False)
        val_loss /= max(len(val_files), 1)

        print(f"  train loss: {train_loss:.6f}  |  val loss: {val_loss:.6f}")

        # -- checkpoint ------------------------------------------------------
        torch.save({
            "epoch":      epoch + 1,
            "model":      classifier.state_dict(),
            "train_loss": train_loss,
            "val_loss":   val_loss,
        }, out_path)

In [ ]:
train_loop(classifier, feat_dir="Z:/users/HadrienTulet/DL_pheno/vjepa/dinov2_Mbal25_patchtokens_reshape", out_path = "Z:/users/HadrienTulet/DL_pheno/vjepa/outputs", T=20, batch_size=4, loss_fn = loss_fn, lr=1e-5, N_epochs= 10)

##### Tests

In [7]:
train_data = np.load("Z:/users/HadrienTulet/DL_pheno/vjepa/dinov2_Mbal25_patchtokens_reshape/chunk_0_features.npy", mmap_mode='r')
train_data = torch.from_numpy(np.transpose(train_data, (1, 0, 2, 3)))
train_data.shape

val_data = np.load("Z:/users/HadrienTulet/DL_pheno/vjepa/dinov2_Mbal25_patchtokens_reshape/chunk_1_features.npy", mmap_mode='r')
val_data = torch.from_numpy(np.transpose(val_data, (1, 0, 2, 3)))
val_data.shape

OSError: [Errno 22] Invalid argument: 'Z:/users/HadrienTulet/DL_pheno/vjepa/dinov2_Mbal25_patchtokens_reshape/chunk_0_features.npy'

In [ ]:
T = 20

feat_dir = "Z:/users/HadrienTulet/DL_pheno/vjepa/dinov2_Mbal25_clstokens_reshape"
features_files = os.listdir(feat_dir)
sep = int(np.ceil(len(features_files)/4))
train_files = features_files[:sep]
val_files = features_files[sep:]

for f in train_files:
    print(f)
    feat_data = np.load(os.path.join(feat_dir, f))
    feat_data = torch.from_numpy(np.transpose(feat_data, (1, 0, 2, 3)))
    for t in range(feat_data.shape[1]-T):
        vec = feat_data[:, t:t+T]
        tgt_vec = feat_data[:, t+T]
        vec = vec.cuda()
        tgt_vec = tgt_vec.cuda()
        res = classifier(vec)
        loss = loss_fn(res, tgt_vec)
        print(res.shape)

for f in val_files:
    print(f)
    feat_data = np.load(os.path.join(feat_dir, f))
    feat_data = torch.from_numpy(np.transpose(feat_data, (1, 0, 2, 3)))
    for t in range(feat_data.shape[1]-T):
        vec = feat_data[:, t:t+T]
        tgt_vec = feat_data[:, t+T]
        vec = vec.cuda()
        tgt_vec = tgt_vec.cuda()
        res = classifier(vec)
        loss = loss_fn(res, tgt_vec)
        print(res.shape)

In [9]:
vec = train_data[0:2, :15]
tgt_vec = train_data[0:2, 15]
vec = vec.to("cuda")
tgt_vec = tgt_vec.to("cuda")

In [5]:
vec.shape

torch.Size([2, 15, 196, 768])

In [8]:
res = classifier(vec)
res.shape

torch.Size([2, 196, 768])